In [ ]:
!pip install transformers
#!pip install -U sentence-transformers


     |████████████████████████████████| 2.9 MB 12.3 MB/s 
     |████████████████████████████████| 3.3 MB 65.5 MB/s 
     |████████████████████████████████| 636 kB 83.7 MB/s 
     |████████████████████████████████| 895 kB 62.7 MB/s 
     |████████████████████████████████| 56 kB 4.9 MB/s 
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 3.13
    Uninstalling PyYAML-3.13:
      Successfully uninstalled PyYAML-3.13


In [ ]:
from transformers import BertTokenizer, BertModel, AutoModel, AutoTokenizer
import pandas as pd
import numpy as np
import torch

In [ ]:
#preprocessing:entry to tokens, map each token to integers
from transformers import AutoTokenizer
#checkpoint = "giacomomiolo/electramed_base_scivocab_1M"
#checkpoint = "google/electra-base-discriminator"
checkpoint = "dmis-lab/biobert-base-cased-v1.2"


model = BertModel.from_pretrained(checkpoint, output_hidden_states = True,)
tokenizer = BertTokenizer.from_pretrained(checkpoint)

Some weights of the model checkpoint at dmis-lab/biobert-base-cased-v1.2 were not used when initializing BertModel: ['cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
def bert_text_preparation(text, tokenizer):
    """Preparing the input for BERT
    
    Takes a string argument and performs
    pre-processing like adding special tokens,
    tokenization, tokens to ids, and tokens to
    segment ids. All tokens are mapped to seg-
    ment id = 1.
    
    Args:
        text (str): Text to be converted
        tokenizer (obj): Tokenizer object
            to convert text into BERT-re-
            adable tokens and ids
        
    Returns:
        list: List of BERT-readable tokens
        obj: Torch tensor with token ids
        obj: Torch tensor segment ids
    
    
    """
    marked_text = "[CLS] " + text + " [SEP]"
    tokenized_text = tokenizer.tokenize(marked_text)
    indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
    segments_ids = [1]*len(indexed_tokens)

    # Convert inputs to PyTorch tensors
    tokens_tensor = torch.tensor([indexed_tokens])
    segments_tensors = torch.tensor([segments_ids])

    return tokenized_text, tokens_tensor, segments_tensors
    
def get_bert_embeddings(tokens_tensor, segments_tensors, model):
    """Get embeddings from an embedding model
    
    Args:
        tokens_tensor (obj): Torch tensor size [n_tokens]
            with token ids for each token in text
        segments_tensors (obj): Torch tensor size [n_tokens]
            with segment ids for each token in text
        model (obj): Embedding model to generate embeddings
            from token and segment ids
    
    Returns:
        list: List of list of floats of size
            [n_tokens, n_embedding_dimensions]
            containing embeddings for each token
    
    """
    
    # Gradient calculation id disabled
    # Model is in inference mode
    with torch.no_grad():
        outputs = model(tokens_tensor, segments_tensors)
        # Removing the first hidden state
        # The first state is the input state
        hidden_states = outputs[2][1:]

    # Getting embeddings from the final BERT layer
    token_embeddings = hidden_states[-1]
    # Collapsing the tensor into 1-dimension
    token_embeddings = torch.squeeze(token_embeddings, dim=0)
    # Converting torchtensors to lists
    list_token_embeddings = [token_embed.tolist() for token_embed in token_embeddings]

    return list_token_embeddings

In [ ]:
from google.colab import files

df=pd.read_csv("./Abstracts-Parsed.csv", sep=',')
#df=pd.read_csv("./Methods-Parsed.csv", sep=',')
df=df.rename(columns={'Content_Parsed': 'text'})
df=df.drop(columns={'Unnamed: 0'})

df.head()

,File_Name,Content,Category,text,Category_Code
0,Gapped Blast,Gapped BLAST and PSI-BLAST: a new generation o...,Alignment,gap blast psiblast new generation protein d...,0
1,RapSearch,RAPSearch: a fast protein similarity search to...,Alignment,rapsearch fast protein similarity search tool...,0
2,PhenoMeter,PhenoMeter: A Metabolome Database Search Tool ...,Alignment,phenometer metabolome database search tool us...,0
3,cuBlASTp,cuBLASTP: Fine-Grained Parallelization of Prot...,Alignment,cublastp finegrained parallelization protein ...,0
4,muBLASTP,muBLASTP: database-indexed protein sequence se...,Alignment,mublastp databaseindexed protein sequence sear...,0


In [ ]:
import unicodedata
pd.options.mode.chained_assignment = None  # default='warn'

import re
re.compile('<title>(.*)</title>')
for i in range (len(df['text'])):
  df['text'][i] = unicodedata.normalize('NFKD', df['text'][i]).encode('ascii', 'ignore').decode("utf-8")
  df['text'][i] = re.sub(r'[^\w]', ' ', df['text'][i])
  df['text'][i] = df['text'][i].encode("ascii", "ignore")
  df['text'][i] = df['text'][i].decode()  

In [ ]:
# Getting embeddings for the target
# word in all given contexts
embeddings = []
all_embeddings = []
sentence_embedding = np.empty(768, dtype=object)
c=0
for text in df["text"]:
    print("*****************item ",c)
    tokenized_text, tokens_tensor, segments_tensors = bert_text_preparation(text, tokenizer)
    list_token_embeddings = get_bert_embeddings(tokens_tensor, segments_tensors, model)
    embeddings.append(list_token_embeddings)
    c=c+1

*****************item  0
*****************item  1
*****************item  2
*****************item  3
*****************item  4
*****************item  5
*****************item  6
*****************item  7
*****************item  8
*****************item  9
*****************item  10
*****************item  11
*****************item  12
*****************item  13
*****************item  14
*****************item  15
*****************item  16
*****************item  17
*****************item  18
*****************item  19
*****************item  20
*****************item  21
*****************item  22
*****************item  23
*****************item  24
*****************item  25
*****************item  26
*****************item  27
*****************item  28
*****************item  29
*****************item  30
*****************item  31
*****************item  32
*****************item  33
*****************item  34
*****************item  35
*****************item  36
*****************item  37
*****************item 

In [ ]:
for l in range(len(embeddings)):
  for v in embeddings[l]:
    sentence_embedding=np.vstack((sentence_embedding, v))
  sentence_embedding = np.delete(sentence_embedding, obj=0, axis=0)
  sentence_embedding = (np.mean(sentence_embedding, axis=0)).tolist()
  all_embeddings.append(sentence_embedding)

all_embeddings = np.array(all_embeddings)
all_embeddings = pd.DataFrame(all_embeddings)

all_embeddings.insert(loc=0, column='text', value=df['text'])

In [ ]:
all_embeddings.head()

,text,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,...,728,729,730,731,732,733,734,735,736,737,738,739,740,741,742,743,744,745,746,747,748,749,750,751,752,753,754,755,756,757,758,759,760,761,762,763,764,765,766,767
0,gap blast psiblast new generation protein d...,0.359182,-0.095408,-0.140700,0.227654,0.098112,-0.208887,-0.131963,0.191976,-0.016109,0.153429,0.206360,0.392494,-0.158565,0.179269,-0.315042,0.094246,0.157331,-0.368918,-0.330913,0.156996,-0.034842,0.169119,0.270348,0.081965,-0.236855,-0.143634,0.231174,0.504116,-0.062666,0.384799,0.143127,0.208104,0.207082,-0.108757,0.009328,-0.071069,-0.069403,0.405751,-0.223823,...,0.081008,0.073380,-0.339893,-0.021166,-0.121670,0.212637,-0.390192,0.182140,0.267565,0.035187,0.488899,-0.184857,-0.352410,-0.232847,0.160319,-0.073348,0.234136,0.395775,-0.311628,0.103759,-0.163563,-0.100477,0.121810,0.429840,0.200505,-0.174230,-0.001749,-0.291837,0.199188,-0.011030,0.117228,0.119529,-0.220935,-0.332495,-0.052891,-0.513627,0.008045,-0.010530,-0.045189,0.525841
1,rapsearch fast protein similarity search tool...,0.378283,-0.102936,-0.223336,0.076764,0.110731,-0.157010,-0.072926,0.205899,-0.068787,0.290040,0.138618,0.288376,-0.346284,0.198446,-0.288311,0.097883,0.140517,-0.384038,-0.343034,-0.037846,-0.130854,0.216447,0.242751,0.103177,-0.189391,-0.142720,0.034726,0.619290,-0.115462,0.199650,0.053907,0.090438,0.093233,-0.103353,-0.004868,-0.046095,-0.161834,0.412565,-0.174754,...,0.068993,0.140696,-0.285975,0.109072,-0.034936,0.471366,-0.159289,0.197146,0.185698,-0.037973,0.476118,-0.195229,-0.217811,-0.508612,-0.040477,0.076247,0.259313,0.474293,-0.144526,-0.018969,-0.016127,-0.042103,0.096182,0.303490,0.289674,-0.028760,-0.110088,-0.427356,0.052555,0.045590,0.149268,0.008431,-0.268572,-0.391621,-0.123869,-0.394320,0.079721,0.169057,0.189928,0.459260
2,phenometer metabolome database search tool us...,0.345957,-0.251558,-0.231275,0.074893,0.008110,-0.080131,-0.177260,0.224882,-0.312352,0.297113,0.403980,0.360522,-0.084134,0.201688,-0.223845,0.158845,0.243522,-0.429495,-0.343213,0.013248,-0.138170,0.103319,0.086296,0.112471,-0.398935,-0.273242,0.030285,0.540476,-0.033157,0.343228,-0.135838,0.274171,0.156809,-0.075478,-0.047389,-0.060295,-0.265549,0.396474,-0.200129,...,0.191757,0.174725,-0.211147,0.087403,-0.287475,0.310549,-0.228212,0.228273,0.373634,0.136052,0.228316,-0.204604,-0.371382,-0.523378,0.401201,0.114103,0.299553,0.377349,-0.204790,-0.049581,-0.053672,-0.058912,0.193861,0.360301,0.340501,-0.098386,0.056639,-0.338500,0.019305,0.037290,-0.130179,0.167868,-0.212788,-0.401717,0.176932,-0.381745,-0.016877,-0.106133,-0.194441,0.541477
3,cublastp finegrained parallelization protein ...,0.425504,-0.075710,-0.146624,0.295694,0.107741,-0.237698,-0.245754,0.282816,-0.130223,0.186961,0.168953,0.337852,-0.197216,0.294017,-0.193529,0.058875,0.211176,-0.316935,-0.489479,-0.101650,0.097471,0.067295,0.346663,0.079177,-0.139805,-0.118854,0.231225,0.622009,-0.127646,0.289882,0.041579,0.441070,0.158586,-0.305112,0.091150,-0.037636,-0.395129,0.583911,-0.053915,...,-0.012448,0.064925,-0.561370,-0.054858,-0.192427,0.532952,-0.333413,0.112347,0.282707,-0.050121,0.264311,-0.028911,-0.256942,-0.368565,0.023481,0.117495,0.207511,0.396442,-0.340336,0.007617,-0.075202,-0.101408,0.073548,0.463564,0.502060,-0.224552,0.020444,-0.363375,0.164245,-0.065981,0.144608,0.130958,-0.406568,-0.396111,-0.061241,-0.411106,-0.043759,0.129102,-0.181024,0.698033
4,mublastp databaseindexed protein sequence sear...,0.439670,-0.173987,-0.140066,0.259406,0.123257,-0.116301,-0.192065,0.361586,-0.183180,0.231872,0.221239,0.376339,-0.014031,0.162585,-0.148981,0.100286,0.054466,-0.301090,-0.561246,0.101803,-0.027969,0.200371,0.200960,0.164427,-0.078767,-0.225102,0.156774,0.711334,-0.047297,0.305893,0.134487,0.270586,0.276677,-0.203819,0.024620,0.082832,-0.208509,0.703147,-0.220087,...,-0.017182,0.151579,-0.410959,0.031416,-0.240859,0.644827,-0.299832,0.142045,0.213147,-

In [ ]:
print(all_embeddings.shape)
print(all_embeddings)
#all_embeddings.to_csv('AbstractsMethodsROBERTAstsb.csv') 
#files.download("AbstractsMethodsROBERTAstsb.csv")
all_embeddings.to_csv('Abstracts-BioBERT-NoSL.csv') 
files.download("Abstracts-BioBERT-NoSL.csv")

(224, 769)
                                                  text  ...       767
0    gap blast  psiblast  new generation  protein d...  ...  0.525841
1    rapsearch  fast protein similarity search tool...  ...  0.459260
2    phenometer  metabolome database search tool us...  ...  0.541477
3    cublastp finegrained parallelization  protein ...  ...  0.698033
4    mublastp databaseindexed protein sequence sear...  ...  0.539436
..                                                 ...  ...       ...
219  quast quality assessment tool  genome assembli...  ...  0.367999
220  versatile genome assembly evaluation  quastlg\...  ...  0.227818
221  busco assess genome assembly  annotation compl...  ...  0.209950
222  dnaqet  framework  compute  consolidate metric...  ...  0.493145
223  laser large genome assembly evaluator\ngenome ...  ...  0.375762

[224 rows x 769 columns]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>